# 03.2 LSTM Intro

`LSTM` is one of the classic sequence models.

Even though many modern scenarios have shifted toward Transformers, `LSTM` is still worth learning because it helps you understand:

- how sequence information propagates across time steps
- hidden state
- the input-output shapes of recurrent models

This notebook demonstrates the full `LSTM` workflow using a small synthetic task.


## Learning Goals

After this notebook, you should be able to:

1. Understand the input and output shapes of `LSTM`.
2. Understand the hidden state and cell state.
3. Write a minimal `LSTM` classifier.
4. Understand what `batch_first=True` means.
5. Train and evaluate an `LSTM` on a small sequence task.
6. Build a contrastive reference point for later attention

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

## First Look at the Shape of a Minimal `LSTM`

Before training anything, first make the shapes crystal clear.

Here we use `batch_first=True`, so the input shape is:

- `(batch_size, seq_len, input_size)`

In [ ]:
lstm = nn.LSTM(input_size=5, hidden_size=7, batch_first=True)
x = torch.randn(4, 6, 5)
output, (h_n, c_n) = lstm(x)

print("x.shape =", x.shape)
print("output.shape =", output.shape)
print("h_n.shape =", h_n.shape)
print("c_n.shape =", c_n.shape)

What this means:

- `output.shape == (4, 6, 7)`
 each time step / time step has a hidden representation
- `h_n.shape == (1, 4, 7)`
  final hidden state
- `c_n.shape == (1, 4, 7)`
  final cell state

## Build a Small Task

To make the `LSTM` learn an actual sequential relation, we build a toy task:

- input is an integer sequence of length 6
- label is whether the first token equals the last token

This task requires the model to remember information from the beginning, so it is more sequence-like than looking at only one position.


In [ ]:
torch.manual_seed(0)

vocab_size = 8
seq_len = 6
num_samples = 1000

X = torch.randint(low=0, high=vocab_size, size=(num_samples, seq_len))
y = (X[:, 0] == X[:, -1]).long()

split = 800
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

print("positive rate / positive rate:", y.float().mean().item())
print("X_train.shape =", X_train.shape)
print("y_train.shape =", y_train.shape)

## Define an `LSTM` Classifier

Common pipeline:

1. token ids -> `Embedding`
2. `Embedding` -> `LSTM`
3. classification head

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        output, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]
        logits = self.fc(last_hidden)
        return logits


model = LSTMClassifier(vocab_size=vocab_size, embed_dim=12, hidden_size=16, num_classes=2)
print(model)

In [ ]:
xb, yb = next(iter(train_loader))
logits = model(xb)

print("xb.shape =", xb.shape)
print("logits.shape =", logits.shape)
print("yb.shape =", yb.shape)

Here `logits.shape == (batch_size, 2)` because this is a binary classification task represented by 2 class scores.


## Training and Evaluation Functions

We continue using the training pattern you already know from earlier notebooks.


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


def batch_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches

## Start Training

We train for 8 epochs here, which is enough to see the `LSTM` learn this small task.


In [ ]:
history = []

for epoch in range(1, 9):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )

    print(
        f"epoch={epoch:02d} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

In [ ]:
print(history[-1])

## Inspect Predictions

Here we directly inspect predictions on a few sequences.


In [ ]:
model.eval()
sample_x = X_val[:8]
sample_y = y_val[:8]

with torch.no_grad():
    sample_logits = model(sample_x)
    sample_preds = sample_logits.argmax(dim=1)

print("sample_x =\n", sample_x)
print("true labels =", sample_y)
print("pred labels =", sample_preds)

## A Shape Exercise

You should now start getting comfortable with the three key shapes around `LSTM`:

- input
- all-step outputs
- final hidden state

In [ ]:
# Exercise 1
# batch_size = 5
# seq_len = 7
# embed_dim = 12
# hidden_size = 16
# batch_first=True
#Questions / Questions:
# 1. What is the input x.shape to the LSTM?
# 2. What is output.shape?
# 
# Answer first by yourself, then check the reference answer.


Reference answer:

- `x.shape == (5, 7, 12)`
- `output.shape == (5, 7, 16)`
- `h_n.shape == (1, 5, 16)`

This assumes a single-layer unidirectional `LSTM`.


In [ ]:
# Exercise 2
# Change the model's hidden_size from 16 to 32, then observe whether logits.shape changes.


Hint:

The final `logits.shape` is determined by the number of output classes, not directly by the hidden size.


## Summary

The most important thing in this notebook is to clearly see the shapes and information flow of the `LSTM`.

You should now be able to answer:

1. Why is the `LSTM` input shape often written as `(B, T, D)`?
2. What is the difference between `output` and `h_n`?
3. Why is the final hidden state often used for classification tasks?

Suggested next step:

- Move to the `Attention` notebook and compare recurrent models with attention-based models.